# Energy Transition — Key Numbers (reproduction notebook)

**Purpose.** Reproduce the Germany Energy Transition dashboard's key verified
numbers and panel statistics **from the project data itself**, so every figure
on the dashboard can be re-checked in one place.

**Sources.** Our World in Data (OWID) energy data and OWID CO₂ data, as stored
in this repo:
- `data/analysis_subset.csv` — Germany, France, Poland, Denmark, 2000–2024
  (25 years × 4 countries).
- `dashboard/powerbi/data/FactCountryYear_Full.csv` — star-schema fact table,
  213 countries.
- `dashboard/powerbi/data/FactGeneration_Full.csv` — generation by source,
  213 countries.
- `data/owid-energy-data.csv` — raw OWID file, used only for the bioenergy /
  oil split that the star schema does not carry separately.

**Rule of this notebook.** Every number printed below is *computed* by the
code cells from those CSVs. The section headings state the dashboard's
verified figure; the printed output is the independent recomputation.
If a printed value ever disagrees with the heading, that is a real
discrepancy to investigate — nothing here is typed in by hand.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# Resolve paths whether the notebook is run from the repo root or from notebooks/
REPO = Path.cwd()
if not (REPO / "data" / "analysis_subset.csv").exists():
    REPO = REPO.parent

sub = pd.read_csv(REPO / "data" / "analysis_subset.csv")                       # 4 countries x 25 years
fcy = pd.read_csv(REPO / "dashboard/powerbi/data/FactCountryYear_Full.csv")   # star schema, 213 countries
fg  = pd.read_csv(REPO / "dashboard/powerbi/data/FactGeneration_Full.csv")    # generation by source
dc  = pd.read_csv(REPO / "dashboard/powerbi/data/DimCountry_Full.csv")
ds  = pd.read_csv(REPO / "dashboard/powerbi/data/DimSource.csv")
owid = pd.read_csv(REPO / "data" / "owid-energy-data.csv",
                    usecols=["country", "year", "biofuel_electricity",
                             "oil_electricity", "other_renewable_exc_biofuel_electricity"])

print(f"analysis_subset.csv:            {len(sub):,} rows  ({sub.country.nunique()} countries x {sub.year.nunique()} years)")
print(f"FactCountryYear_Full.csv:       {len(fcy):,} rows")
print(f"FactGeneration_Full.csv:        {len(fg):,} rows")
print(f"DimCountry_Full.csv:            {dc.Country.nunique()} countries")
print(f"Years covered:                  {sub.year.min()}–{sub.year.max()} (subset), "
      f"{fcy.YearKey.min()}–{fcy.YearKey.max()} (star schema)")

def ols(x, y):
    """Ordinary least squares y ~ x. Returns (slope, intercept, R^2)."""
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    A = np.vstack([x, np.ones_like(x)]).T
    slope, intercept = np.linalg.lstsq(A, y, rcond=None)[0]
    pred = slope * x + intercept
    r2 = 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    return slope, intercept, r2

def cagr(start, end, years):
    return (end / start) ** (1 / years) - 1

de = sub[sub.country == "Germany"].sort_values("year").reset_index(drop=True)
print("\nGermany rows:", len(de), "| France/Poland/Denmark rows:",
      *(len(sub[sub.country == c]) for c in ["France", "Poland", "Denmark"]))

analysis_subset.csv:            100 rows  (4 countries x 25 years)
FactCountryYear_Full.csv:       5,271 rows
FactGeneration_Full.csv:        36,897 rows
DimCountry_Full.csv:            213 countries
Years covered:                  2000–2024 (subset), 2000–2024 (star schema)

Germany rows: 25 | France/Poland/Denmark rows: 25 25 25


## 1. Headline — Germany's renewable share

Dashboard claim: renewable share **6.22% (2000) → 58.64% (2024)**; OLS trend
**+2.05 pp/yr, R² = 0.95**; first year above 50% = **2023**; at the linear
trend, 80% is reached in **2039**.

In [2]:
share_2000 = de.loc[de.year == 2000, "renewables_share_elec"].iloc[0]
share_2024 = de.loc[de.year == 2024, "renewables_share_elec"].iloc[0]
slope, intercept, r2 = ols(de.year, de.renewables_share_elec)
first_over_50 = de.loc[de.renewables_share_elec > 50, "year"].min()
year_80 = int(np.ceil((80 - intercept) / slope))

print(f"Renewable share 2000: {share_2000:.2f}%")
print(f"Renewable share 2024: {share_2024:.2f}%")
print(f"Change 2000->2024:     {share_2024 - share_2000:+.2f} percentage points")
print(f"OLS trend:             {slope:+.2f} pp/yr, R² = {r2:.2f}")
print(f"First year > 50%:      {first_over_50}")
print(f"Linear projection:     80% reached in {year_80}  (exact: {(80 - intercept) / slope:.1f})")

Renewable share 2000: 6.22%
Renewable share 2024: 58.64%
Change 2000->2024:     +52.42 percentage points
OLS trend:             +2.05 pp/yr, R² = 0.95
First year > 50%:      2023
Linear projection:     80% reached in 2039  (exact: 2038.8)


## 2. Wind and solar — the growth engine

Dashboard claim: wind **9.35 → 141.6 TWh (CAGR 12.0%)**; solar **0.1 TWh
(2001) → 74.1 TWh (CAGR 33.3%)**; wind + solar = **80.8%** of all renewable
growth since 2000. (Solar was 0.0 TWh in 2000, so its CAGR is measured from
the first non-zero year, 2001.)

In [3]:
wind_2000, wind_2024 = de.loc[de.year == 2000, "wind_electricity"].iloc[0], de.loc[de.year == 2024, "wind_electricity"].iloc[0]
solar_2001 = de.loc[de.year == 2001, "solar_electricity"].iloc[0]
solar_2024 = de.loc[de.year == 2024, "solar_electricity"].iloc[0]
ren_2000, ren_2024 = de.loc[de.year == 2000, "renewables_electricity"].iloc[0], de.loc[de.year == 2024, "renewables_electricity"].iloc[0]

print(f"Wind  2000: {wind_2000:.2f} TWh  -> 2024: {wind_2024:.1f} TWh  (x{wind_2024 / wind_2000:.1f}, CAGR {cagr(wind_2000, wind_2024, 24) * 100:.1f}%)")
print(f"Solar 2001: {solar_2001:.1f} TWh  -> 2024: {solar_2024:.1f} TWh  (x{solar_2024 / solar_2001:.0f}, CAGR {cagr(solar_2001, solar_2024, 23) * 100:.1f}%)")
ws_growth = (wind_2024 + solar_2024) - (wind_2000 + de.loc[de.year == 2000, 'solar_electricity'].iloc[0])
print(f"Wind+solar share of renewable growth: {ws_growth / (ren_2024 - ren_2000) * 100:.1f}%")
print(f"  (renewables {ren_2000:.2f} -> {ren_2024:.2f} TWh; wind+solar added {ws_growth:.2f} TWh)")

Wind  2000: 9.35 TWh  -> 2024: 141.6 TWh  (x15.1, CAGR 12.0%)
Solar 2001: 0.1 TWh  -> 2024: 74.1 TWh  (x741, CAGR 33.3%)
Wind+solar share of renewable growth: 80.8%
  (renewables 35.41 -> 290.86 TWh; wind+solar added 206.38 TWh)


## 3. Coal and nuclear — what renewables replaced

Dashboard claim: coal share **52.1% → 21.4%**; first year renewables beat
coal = **2019**; nuclear **29.8% → 0%** (last plants closed April 2023).

In [4]:
coal_2000, coal_2024 = de.loc[de.year == 2000, "coal_share_elec"].iloc[0], de.loc[de.year == 2024, "coal_share_elec"].iloc[0]
nuc_2000, nuc_2024 = de.loc[de.year == 2000, "nuclear_share_elec"].iloc[0], de.loc[de.year == 2024, "nuclear_share_elec"].iloc[0]
first_beats_coal = de.loc[de.renewables_share_elec > de.coal_share_elec, "year"].min()

print(f"Coal share:    {coal_2000:.1f}% (2000) -> {coal_2024:.1f}% (2024), {coal_2024 - coal_2000:+.1f} pp")
print(f"Nuclear share: {nuc_2000:.1f}% (2000) -> {nuc_2024:.1f}% (2024)")
print(f"First year renewables beat coal: {first_beats_coal}")
print(f"  ({first_beats_coal}: renewables {de.loc[de.year == first_beats_coal, 'renewables_share_elec'].iloc[0]:.1f}% "
      f"vs coal {de.loc[de.year == first_beats_coal, 'coal_share_elec'].iloc[0]:.1f}%)")

Coal share:    52.1% (2000) -> 21.4% (2024), -30.7 pp
Nuclear share: 29.8% (2000) -> 0.0% (2024)
First year renewables beat coal: 2019
  (2019: renewables 40.2% vs coal 28.6%)


## 4. Carbon — cleaner electricity

Dashboard claim: carbon intensity **573.2 → 336.4 gCO₂/kWh (−41.3%)**;
renewable share vs carbon intensity, Germany: Pearson **r = −0.95**, OLS
slope **−4.6 gCO₂/kWh per pp**, **R² = 0.91**; CO₂ per capita **10.99 →
6.79 t (−38.2%)**.

In [5]:
ci_2000, ci_2024 = de.loc[de.year == 2000, "carbon_intensity_elec"].iloc[0], de.loc[de.year == 2024, "carbon_intensity_elec"].iloc[0]
co2_2000, co2_2024 = de.loc[de.year == 2000, "co2_per_capita"].iloc[0], de.loc[de.year == 2024, "co2_per_capita"].iloc[0]
r = np.corrcoef(de.renewables_share_elec, de.carbon_intensity_elec)[0, 1]
slope_ci, _, r2_ci = ols(de.renewables_share_elec, de.carbon_intensity_elec)

print(f"Carbon intensity: {ci_2000:.1f} -> {ci_2024:.1f} gCO2/kWh  ({(ci_2024 / ci_2000 - 1) * 100:+.1f}%)")
print(f"CO2 per capita:   {co2_2000:.2f} -> {co2_2024:.2f} t  ({(co2_2024 / co2_2000 - 1) * 100:+.1f}%)")
print(f"Renewable share vs carbon intensity: Pearson r = {r:.2f}")
print(f"OLS: carbon intensity falls {abs(slope_ci):.1f} gCO2/kWh per +1 pp renewable share (R² = {r2_ci:.2f})")

Carbon intensity: 573.2 -> 336.4 gCO2/kWh  (-41.3%)
CO2 per capita:   10.99 -> 6.79 t  (-38.2%)
Renewable share vs carbon intensity: Pearson r = -0.95
OLS: carbon intensity falls 4.6 gCO2/kWh per +1 pp renewable share (R² = 0.91)


## 5. Q7/Q8 — Germany among the large producers

Dashboard claim (2024, producers with ≥100 TWh generation): Germany ranks
**#5 of 36** by renewable share; the fastest transition of all 36 is
Germany's **+52.4 pp** since 2000.

In [6]:
f24 = fcy[fcy.YearKey == 2024]
f00 = fcy[fcy.YearKey == 2000].set_index("CountryKey")
large = f24[f24.TotalGeneration_TWh >= 100].copy()
large["Change_pp"] = large.RenewableShare_Pct - large.CountryKey.map(f00.RenewableShare_Pct)
large = large.merge(dc, on="CountryKey").sort_values("RenewableShare_Pct", ascending=False).reset_index(drop=True)
large["Rank"] = large.index + 1

print(f"Large producers (>=100 TWh) in 2024: {len(large)}")
print(large[["Rank", "Country", "TotalGeneration_TWh", "RenewableShare_Pct"]]
      .head(10).rename(columns={"TotalGeneration_TWh": "Gen_TWh", "RenewableShare_Pct": "RenShare_%"})
      .to_string(index=False, float_format=lambda v: f"{v:.1f}"))
de_rank = int(large.loc[large.Country == "Germany", "Rank"].iloc[0])
print(f"\nGermany rank: #{de_rank}  (share {large.loc[large.Country == 'Germany', 'RenewableShare_Pct'].iloc[0]:.1f}%)")
fastest = large.sort_values("Change_pp", ascending=False).iloc[0]
print(f"Fastest transition 2000->2024: {fastest.Country} {fastest.Change_pp:+.1f} pp "
      f"(#{int(large.sort_values('Change_pp', ascending=False).reset_index(drop=True).loc[lambda d: d.Country == 'Germany'].index[0]) + 1} overall is Germany)")

Large producers (>=100 TWh) in 2024: 36
 Rank        Country  Gen_TWh  RenShare_%
    1         Norway    157.1        98.6
    2         Brazil    745.7        87.3
    3         Sweden    172.2        69.4
    4         Canada    636.8        64.3
    5        Germany    496.0        58.6
    6          Spain    280.9        57.3
    7    Netherlands    122.3        50.7
    8 United Kingdom    284.2        50.6
    9          Italy    267.6        50.2
   10         Turkey    341.2        45.6

Germany rank: #5  (share 58.6%)
Fastest transition 2000->2024: Germany +52.4 pp (#1 overall is Germany)


## 6. Q15 — the 90%+ club

Dashboard claim (2024, all 195 countries in the star schema): **20**
countries run on ≥90% renewables; **7** are at exactly 100%; Germany ranks
**#55 of 195**.

In [7]:
f24 = fcy[fcy.YearKey == 2024].merge(dc, on="CountryKey")
n_ge90 = int((f24.RenewableShare_Pct >= 90).sum())
at_100 = sorted(f24.loc[f24.RenewableShare_Pct == 100, "Country"].tolist())
de_share = f24.loc[f24.Country == "Germany", "RenewableShare_Pct"].iloc[0]
de_rank_all = int((f24.RenewableShare_Pct > de_share).sum()) + 1

print(f"Countries in 2024: {len(f24)}")
print(f"Countries >= 90% renewable: {n_ge90}")
print(f"Countries at exactly 100%: {len(at_100)} — {', '.join(at_100)}")
print(f"Germany: {de_share:.1f}% renewable -> rank #{de_rank_all} of {len(f24)}")

Countries in 2024: 195
Countries >= 90% renewable: 20
Countries at exactly 100%: 7 — Albania, Bhutan, Democratic Republic of Congo, Ethiopia, Iceland, Nepal, Paraguay
Germany: 58.6% renewable -> rank #55 of 195


## 7. Q16 — heatmap: the 12 largest producers over time

Renewable share (%) for the 12 largest electricity producers in
2000 / 2005 / 2010 / 2015 / 2020 / 2024, from the star-schema fact table.

In [8]:
big12 = ["China", "United States", "India", "Russia", "Japan", "Brazil",
         "Canada", "South Korea", "France", "Germany", "Saudi Arabia", "Iran"]
piv = (fcy.merge(dc, on="CountryKey")
          .pivot_table(index="Country", columns="YearKey", values="RenewableShare_Pct"))
print(piv.loc[big12, [2000, 2005, 2010, 2015, 2020, 2024]].round(1).to_string())
print("\nGermany row again, for the record: "
      f"{piv.loc['Germany', 2000]:.1f}% (2000) -> {piv.loc['Germany', 2024]:.1f}% (2024)")

YearKey        2000  2005  2010  2015  2020  2024
Country                                          
China          16.6  16.2  18.7  24.0  28.1  33.7
United States   9.2   8.7  10.3  13.6  20.3  24.1
India          14.0  15.3  15.1  14.6  19.0  19.8
Russia         18.7  18.1  16.1  15.8  19.9  18.1
Japan           9.2   8.9  10.2  15.0  19.1  22.9
Brazil         89.5  87.1  84.8  74.2  83.2  87.3
Canada         60.6  60.1  61.1  63.8  66.6  64.3
South Korea     1.4   1.0   1.1   2.8   7.0   9.7
France         12.7   9.7  13.6  16.0  23.7  27.2
Germany         6.2  10.3  16.9  29.4  44.4  58.6
Saudi Arabia    0.0   0.0   0.0   0.0   0.2   2.2
Iran            3.2   8.2   4.4   4.8   7.4   5.3

Germany row again, for the record: 6.2% (2000) -> 58.6% (2024)


## 8. Q17 — bridge: Germany's generation mix, 2000 vs 2024

Dashboard claim: total generation **569.0 TWh (2000) → 496.0 TWh (2024)**,
closing check **569.0 + (−73.0) = 496.0 TWh**.

**Note on definitions.** The dashboard waterfall splits bioenergy
(**+46.75 TWh**, OWID `biofuel_electricity`) and oil & other (**+2.56 TWh**,
oil + other renewables excl. biofuel) as separate bars. The star schema has
no separate bioenergy/oil sources: its **"Other Renewables"** residual is
renewables − hydro − wind − solar, i.e. bioenergy + other renewables
(**+46.97 TWh**). The 0.22 TWh gap is exactly the other-renewables
(geothermal etc.) component. Both splits are shown below and reconcile.
The star schema also carries **no Oil source at all**, so its Germany totals
(550.90 → 475.60 TWh) sit below the true totals by the oil generation
(18.05 → 20.39 TWh); the bridge below therefore uses the subset totals plus
the raw OWID columns, and the star-schema check is printed for transparency.

In [9]:
owg = owid[(owid.country == "Germany") & (owid.year.isin([2000, 2024]))].set_index("year")
de_idx = de.set_index("year")
rows = [("Coal", "coal_electricity", None), ("Gas", "gas_electricity", None),
        ("Nuclear", "nuclear_electricity", None), ("Hydro", "hydro_electricity", None),
        ("Wind", "wind_electricity", None), ("Solar", "solar_electricity", None),
        ("Bioenergy", None, "biofuel_electricity"),
        ("Oil", None, "oil_electricity"),
        ("Other renewables", None, "other_renewable_exc_biofuel_electricity")]
bridge = []
for name, sub_col, ow_col in rows:
    v00 = de_idx.loc[2000, sub_col] if sub_col else owg.loc[2000, ow_col]
    v24 = de_idx.loc[2024, sub_col] if sub_col else owg.loc[2024, ow_col]
    bridge.append((name, v00, v24, v24 - v00))
b = pd.DataFrame(bridge, columns=["Source", "2000_TWh", "2024_TWh", "Delta_TWh"]).set_index("Source")
print(b.round(2).to_string())
tot00 = de_idx.loc[2000, "electricity_generation"]; tot24 = de_idx.loc[2024, "electricity_generation"]
print(f"\nTotal: {tot00:.2f} -> {tot24:.2f} TWh   (delta {tot24 - tot00:+.2f} TWh)")
print(f"Closing check: {tot00:.1f} + ({tot24 - tot00:+.1f}) = {tot24:.1f} TWh  |  sum of source deltas: {b.Delta_TWh.sum():+.2f} TWh")

# Star-schema reconciliation
g = fg[fg.CountryKey == 75].merge(ds, on="SourceKey")
pivg = g.pivot_table(index="Source", columns="YearKey", values="Generation_TWh")
other_delta = pivg.loc["Other Renewables", 2024] - pivg.loc["Other Renewables", 2000]
print(f"\nStar-schema check: FactGeneration Germany total {pivg[2000].sum():.2f} -> {pivg[2024].sum():.2f} TWh")
print(f"  (below the true totals by oil: {owg.loc[2000, 'oil_electricity']:.2f} / {owg.loc[2024, 'oil_electricity']:.2f} TWh — no Oil source in the schema)")
print(f"  'Other Renewables' residual delta: {other_delta:+.2f} TWh = bioenergy "
      f"{b.loc['Bioenergy', 'Delta_TWh']:+.2f} + other renewables {b.loc['Other renewables', 'Delta_TWh']:+.2f}")
print(f"  Waterfall 'oil & other' bar: oil {b.loc['Oil', 'Delta_TWh']:+.2f} + other {b.loc['Other renewables', 'Delta_TWh']:+.2f} "
      f"= {b.loc['Oil', 'Delta_TWh'] + b.loc['Other renewables', 'Delta_TWh']:+.2f} TWh")

                  2000_TWh  2024_TWh  Delta_TWh
Source                                         
Coal                296.68    106.33    -190.35
Gas                  49.20     78.41      29.21
Nuclear             169.61      0.00    -169.61
Hydro                21.73     23.83       2.10
Wind                  9.35    141.60     132.25
Solar                 0.00     74.13      74.13
Bioenergy             4.33     51.08      46.75
Oil                  18.05     20.39       2.34
Other renewables      0.00      0.22       0.22

Total: 568.95 -> 495.99 TWh   (delta -72.96 TWh)
Closing check: 569.0 + (-73.0) = 496.0 TWh  |  sum of source deltas: -72.96 TWh

Star-schema check: FactGeneration Germany total 550.90 -> 475.60 TWh
  (below the true totals by oil: 18.05 / 20.39 TWh — no Oil source in the schema)
  'Other Renewables' residual delta: +46.97 TWh = bioenergy +46.75 + other renewables +0.22
  Waterfall 'oil & other' bar: oil +2.34 + other +0.22 = +2.56 TWh


## 9. Q19 — waffle: Germany's 2024 mix in 100 squares

Dashboard claim — 2024 shares: Wind **28.5**, Coal **21.4**, Gas **15.8**,
Solar **14.9**, Bioenergy **10.3**, Hydro **4.8**, Oil & other **4.2**
(percent, sums to ~100). The waffle uses the **largest-remainder method**:
floor each share, then hand the leftover squares to the largest fractional
remainders.

In [10]:
de24 = de.set_index("year").loc[2024]
ow24 = owid[(owid.country == "Germany") & (owid.year == 2024)].iloc[0]
tot = de24.electricity_generation
shares = {
    "Wind": de24.wind_electricity / tot * 100,
    "Coal": de24.coal_electricity / tot * 100,
    "Gas": de24.gas_electricity / tot * 100,
    "Solar": de24.solar_electricity / tot * 100,
    "Bioenergy": ow24.biofuel_electricity / tot * 100,
    "Hydro": de24.hydro_electricity / tot * 100,
    "Oil & other": (ow24.oil_electricity + ow24.other_renewable_exc_biofuel_electricity) / tot * 100,
}
print("2024 shares (%):")
for k, v in shares.items():
    print(f"  {k:<13} {v:.1f}")

floors = {k: int(np.floor(v)) for k, v in shares.items()}
leftover = 100 - sum(floors.values())
remainders = sorted(shares, key=lambda k: shares[k] - floors[k], reverse=True)
squares = dict(floors)
for k in remainders[:leftover]:
    squares[k] += 1
print(f"\nLargest-remainder squares (leftover {leftover} squares to "
      f"{', '.join(remainders[:leftover])}):")
print("  " + "  ".join(f"{k} {squares[k]}" for k in shares))
print(f"  Total squares: {sum(squares.values())}")

2024 shares (%):
  Wind          28.5
  Coal          21.4
  Gas           15.8
  Solar         14.9
  Bioenergy     10.3
  Hydro         4.8
  Oil & other   4.2

Largest-remainder squares (leftover 4 squares to Solar, Gas, Hydro, Wind):
  Wind 29  Coal 21  Gas 16  Solar 15  Bioenergy 10  Hydro 5  Oil & other 4
  Total squares: 100


## 10. Panel statistics — all four countries

Per-country OLS trend of renewable share (slope pp/yr, R²); correlation and
OLS slope of carbon intensity on renewable share; fossil + nuclear share
(coal + gas + nuclear generation ÷ total generation) 2000 → 2024; and the
cross-country correlation (2024, the 36 large producers) between renewable
share and carbon intensity.

In [11]:
print(f"{'Country':<9} {'Trend slope':>11} {'R²':>5}  |  {'r (share vs CI)':>15} {'CI slope':>8}  |  {'Fossil+nuclear':>15}")
for c in ["Germany", "France", "Poland", "Denmark"]:
    d = sub[sub.country == c].sort_values("year").reset_index(drop=True)
    slope, _, r2 = ols(d.year, d.renewables_share_elec)
    r = np.corrcoef(d.renewables_share_elec, d.carbon_intensity_elec)[0, 1]
    c_slope, _, _ = ols(d.renewables_share_elec, d.carbon_intensity_elec)
    fn = lambda row: (row.coal_electricity + row.gas_electricity + row.nuclear_electricity) / row.electricity_generation * 100
    fn00, fn24 = fn(d.iloc[0]), fn(d.iloc[-1])
    print(f"{c:<9} {slope:+11.2f} {r2:5.2f}  |  {r:15.2f} {c_slope:8.1f}  |  {fn00:5.1f}% -> {fn24:4.1f}%")

large24 = fcy[(fcy.YearKey == 2024) & (fcy.TotalGeneration_TWh >= 100)]
r_cross = np.corrcoef(large24.RenewableShare_Pct, large24.CarbonIntensity_gCO2_kWh)[0, 1]
print(f"\nCross-country (2024, {len(large24)} large producers): "
      f"renewable share vs carbon intensity, r = {r_cross:.2f}")

Country   Trend slope    R²  |  r (share vs CI) CI slope  |   Fossil+nuclear
Germany         +2.05  0.95  |            -0.95     -4.6  |   90.6% -> 37.2%
France          +0.64  0.82  |            -0.75     -1.6  |   85.2% -> 71.1%
Poland          +1.06  0.89  |            -0.99    -11.6  |   95.6% -> 66.2%
Denmark         +3.45  0.96  |            -1.00     -6.8  |   71.0% ->  7.6%

Cross-country (2024, 36 large producers): renewable share vs carbon intensity, r = -0.81
